In [4]:
import numpy as np 
import pandas as pd 
import keras 
from hyperopt import STATUS_OK,Trials,fmin,hp,tpe
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.models import infer_signature

c:\Users\Edunet Foundation\Desktop\MLOPS\mlflowvenv\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
c:\Users\Edunet Foundation\Desktop\MLOPS\mlflowvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data=pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [3]:
#split the data into train and test
train, test = train_test_split(data,test_size=0.2, random_state=42)

In [5]:
train_x=train.drop(['quality'],axis=1).values
train_y=train[['quality']].values.ravel()

## test dataset
test_x=test.drop(['quality'],axis=1).values
test_y=test[['quality']].values.ravel()

## splitting this train data into train and validation

train_x,valid_x,train_y,valid_y=train_test_split(train_x,train_y,test_size=0.20,random_state=42)

signature=infer_signature(train_x,train_y)

In [1]:
#writing the function for model training


def model_training(params,epochs,train_x,train_y,valid_x,valid_y,test_x,test_y):

    mean = np.mean(train_x,axis=0)
    var = np.var(train_x,axis=0)

    #define the neural network
    model = keras.Sequential([
        keras.Input(train_x.shape[1]),
        keras.layers.Normalization(mean=mean,variance=var),
        keras.layers.Dense(64,activation=relu),
        keras.layers.Dense(1)
    ])

    #model Compilation
    model.compile(optimizer=keras.optimizers.SGD(
        learning_rate=params['lr'],momentum=params['momentum']),
        loss='mean_squared_error',metrics=[keras.metrics.RootMeanSquaredError])


    #now we will start our training as well as logs for mlflow

    mlflow.set_tracking_uri('http://127.0.0.1/5000')

    with mlflow.start_run(nested=True):
        model.fit(train_x,train_y,epochs=epochs,validation_data=(valid_x,valid_y),batch_size=64)

        ev_result = model.evaluate(valid_x,valid_y,batch_size=64)

        ev_rmse = ev_result[1]

        mlflow.log_params(params)
        mlflow.log_metric('rmse',ev_rmse)

        #log the model as well
        mlflow.tensorflow.log_model('model',model,signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

In [6]:
def objective(params):
    # MLflow will track the parameters and results for each run
    result = model_training(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
    )
    return result

In [7]:
#setup the space for hyperparameter Tuning
space={
    "lr":hp.loguniform("lr",np.log(1e-5),np.log(1e-1)),
    "momentum":hp.uniform("momentum",0.0,1.0)

}

In [ ]:
mlflow.set_tracking_uri('http://127.0.0.1/5000')
mlflow.set_experiment('/wine-quality')
with mlflow.start_run():
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )

    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

PermissionError: [WinError 5] Access is denied: 'C:\\Users\\Edunet%20Foundation'